# Facial emotion recognition: CK+ stage on Colab

Trains the three models (`cnn_scratch`, `resnet18`, `mobilenet_v2`) on **CK+** (927 images, 106 people, 6 classes: no neutral, contempt dropped), measures their accuracy with **subject-independent 10-fold cross-validation**, and keeps one CK+ checkpoint per model and seed for the **transfer** to the target dataset (section 5, once that dataset is ready). Same code and configs as the repository; see `CLAUDE.md` §3.6.

**Before running**
1. On the PC: `python colab/make_bundle.py`, then upload `colab/upload/facial_emotion_colab.zip` to `MyDrive/facial_emotion/`.
2. Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

**Outputs** are copied to `MyDrive/facial_emotion/runs/` and `MyDrive/facial_emotion/reports/` after every block. If the session disconnects, run all cells again: finished runs are skipped.

Time on a T4: about 20 min for section 2 and about 3 h for section 3 (90 runs).

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import collections, glob, json, os, re, subprocess
import pandas as pd
from IPython.display import Image, Markdown, display

DRIVE = '/content/drive/MyDrive/facial_emotion'   # bundle in, results out
WORK = '/content/facial'                          # code and data on the fast local disk
BUNDLE = f'{DRIVE}/facial_emotion_colab.zip'
assert os.path.exists(BUNDLE), f'upload facial_emotion_colab.zip to {DRIVE}/ first'

subprocess.run(f'rm -rf {WORK} && mkdir -p {WORK} && unzip -q "{BUNDLE}" -d {WORK}', shell=True, check=True)
os.chdir(WORK)
os.makedirs(f'{DRIVE}/runs', exist_ok=True)
os.makedirs(f'{DRIVE}/reports', exist_ok=True)
# finished runs of earlier sessions come back, so the blocks below skip them
subprocess.run('which rsync > /dev/null || apt-get -qq install -y rsync > /dev/null', shell=True, check=True)
subprocess.run(f'rsync -a "{DRIVE}/runs/" runs/', shell=True, check=True)
os.environ['PYTHONUTF8'] = '1'
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
MODELS = ['cnn_scratch', 'resnet18', 'mobilenet_v2']
SEEDS = [42, 123, 2024]


def sh(cmd, show=None):
    # Run a command in the repository; print the lines matching `show` (all if None).
    # On failure print the last 40 lines and stop.
    tail = collections.deque(maxlen=40)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        tail.append(line)
        if show is None or re.search(show, line):
            print(line, end='', flush=True)
    if p.wait():
        print(''.join(tail))
        raise RuntimeError(f'exit {p.returncode}: {cmd}')


def sync():
    # Copy runs and reports to Drive. last.pt and the 90 CV fold checkpoints (~2 GB) stay on
    # the Colab disk; the CV summary only needs their test_probs.npz, config.yaml and log.csv.
    os.makedirs('reports', exist_ok=True)
    sh(f'rsync -a --exclude last.pt --exclude "ckplus_cv/*/*.pt" --exclude "dryrun_*" runs/ "{DRIVE}/runs/"')
    sh(f'rsync -a reports/ "{DRIVE}/reports/"')


def latest(pattern):
    # Directory of the newest file matching `pattern`, or None.
    hits = sorted(glob.glob(pattern))
    return os.path.dirname(hits[-1]) if hits else None

## 1. Sanity checks
`--dry-run` (2 batches per stage) and `--overfit-batch` (one batch must reach 100% accuracy) for each CK+ config.

In [ ]:
for m in MODELS:
    sh(f'python -m src.train --config configs/ckplus/{m}.yaml --dry-run', show=r'^(done|stage)')
    sh(f'python -m src.train --config configs/ckplus/{m}.yaml --overfit-batch', show=r'^overfit-batch (PASS|FAIL)')

## 2. Transfer checkpoints (3 models × 3 seeds)
CK+ people keep the split they have in `dataset/` (train 84 people / 756 images, val 16 / 126, test 6 / 45), so these models never saw a val or test person of `dataset/`; they are the starting point of section 5. Each is evaluated once on its 45 test images. That test set has no *fear* image, so its macro-F1 over 6 classes counts fear as 0: read the accuracy there, and section 3 for the real evidence.

In [ ]:
for m in MODELS:
    for s in SEEDS:
        if latest(f'runs/ckplus/{m}_s{s}_*/test_metrics.json'):
            print(f'{m} seed {s}: done')
            continue
        run = latest(f'runs/ckplus/{m}_s{s}_*/curves.png')
        if run is None or not os.path.exists(f'{run}/best.pt'):
            sh(f'python -m src.train --config configs/ckplus/{m}.yaml --seed {s}', show=r'^(run dir|done)')
            run = latest(f'runs/ckplus/{m}_s{s}_*/curves.png')
        sh(f'python -m src.evaluate --run {run}', show=r'on test|  accuracy ')
        sync()

rows = []
for f in sorted(glob.glob('runs/ckplus/*/test_metrics.json')):
    r = json.load(open(f))
    rows.append({'model': r['model'], 'seed': r['seed'], 'run': r['run'], 'best val macro-F1': r['best_val_macro_f1'],
                 'test accuracy (45 images)': r['accuracy']})
display(pd.DataFrame(rows).round(4))

## 3. Subject-independent 10-fold cross-validation (3 models × 3 seeds × 10 folds)
Fold k: test = fold k, val = fold k+1 (checkpoint choice), train = the other 8 folds; every person is in one fold only. Each block (one model, one seed: 10 runs) is copied to Drive when it ends, so a disconnect loses at most one block.

In [ ]:
for m in MODELS:
    for s in SEEDS:
        sh(f'python -m src.ckplus_cv --models {m} --seeds {s}', show=r'^\[')
        sync()

## 4. Summary

In [ ]:
sh('python -m src.ckplus_cv --summary-only', show='(?!)')   # writes reports/ckplus_cv.md and the figures (shown below)
sync()
display(Markdown(open('reports/ckplus_cv.md', encoding='utf-8').read()))
for m in MODELS:
    display(Image(f'reports/figures/ckplus_cv_confusion_{m}.png', width=420))

## 5. Transfer to the VTV target dataset (CLAUDE.md §3.7)
1. Upload the dataset to `MyDrive/facial_emotion/` as `target_dataset.zip` (`dataset/<split>/<class>/*.png` + `dataset/manifest.csv`, written by `src.data.vtv_build`), or as its parts `target_dataset.zip.part00`, `part01`, … (browser uploads ≤ 10 MB; made with `split -b 9m -d target_dataset.zip target_dataset.zip.part`). The parts are joined here.
2. The VTV faces have no CK+ person, so the CK+ checkpoints of section 2 never saw a val or test person of the target dataset: no rebuild of `data/ckplus/` is needed.
3. `TAG` names the label version: runs go to `runs/transfer_<TAG>/`, the summary to `reports/transfer_<TAG>_summary.md`. **`vtv_suggested` = labels are model suggestions** (FER+ + HSEmotion ensemble and Claude's agreements, no human review; user's decision 2026-11-07 to train anyway): its test "accuracy" is the agreement with those labels, not a measured accuracy. **`vtv_claude`** (2026-11-08, user: Claude applies its own screen) = the same suggestions after Claude's visual pre-screen of disgust, fear, surprise, angry (1,844 faces moved to `_reject`/`_unsure`), uploaded as `target_dataset_vtv_claude.zip.part*`: still not human-checked, so its numbers are agreement too. After the human review, rebuild, upload again and run with `TAG = 'vtv_reviewed'`.
4. Only setup (section 0, both cells) and this cell are needed: run them one by one (not Run all). About 5–10 min per run on a T4 (9 runs).

In [ ]:
RUN_TRANSFER = True
TAG = 'vtv_claude'      # label version (see above); its zip: target_dataset_<TAG>.zip(.partNN), else target_dataset.zip

if RUN_TRANSFER:
    name = f'target_dataset_{TAG}.zip'
    if not glob.glob(f'{DRIVE}/{name}*'):
        name = 'target_dataset.zip'   # the first upload (vtv_suggested) has no tag in its name
    parts = sorted(glob.glob(f'{DRIVE}/{name}.part*'))
    zip_path = f'{DRIVE}/{name}'
    if parts:   # uploaded in < 10 MB parts: join them on the local disk
        zip_path = '/content/target_dataset.zip'
        sh('cat ' + ' '.join(f'"{p}"' for p in parts) + f' > {zip_path}')
        print(f'joined {len(parts)} parts of {name}: {os.path.getsize(zip_path) / 2**20:.1f} MB')
    sh(f'rm -rf {WORK}/dataset && unzip -q -o "{zip_path}" -d {WORK}')   # unzip checks every CRC
    sh('python -m src.data.index_splits')   # dataset/ -> data/splits/*.csv (the transfer configs read these)
    runs_dir = f'runs/transfer_{TAG}'
    for m in MODELS:
        cfg = f'configs/transfer/{m}_{TAG}.yaml'   # same config, own runs_dir per label version
        with open(cfg, 'w') as f:
            f.write(f'_base_: {m}.yaml\nruns_dir: {runs_dir}\n')
        for s in SEEDS:
            if latest(f'{runs_dir}/{m}_s{s}_*/test_metrics.json'):
                print(f'{m} seed {s}: done')
                continue
            run = latest(f'{runs_dir}/{m}_s{s}_*/curves.png')
            if run is None or not os.path.exists(f'{run}/best.pt'):
                init = latest(f'runs/ckplus/{m}_s{s}_*/test_metrics.json')   # the finished CK+ run (section 2)
                assert init, f'no finished CK+ run for {m} seed {s} (section 2)'
                sh(f'python -m src.train --config {cfg} --seed {s} --init {init} --dry-run', show=r'^(init from|done)')
                sh(f'python -m src.train --config {cfg} --seed {s} --init {init}', show=r'^(run dir|epoch|done)')
                run = latest(f'{runs_dir}/{m}_s{s}_*/curves.png')
            sh(f'python -m src.evaluate --run {run}', show=r'on test|  accuracy ')
            sync()

    sh(f'python -m src.aggregate --runs {runs_dir} --out reports/transfer_{TAG}_summary.csv', show='(?!)')
    sync()
    note = {'vtv_suggested': '**Test labels are model suggestions (not human-checked): accuracy = agreement with them.**\n\n',
            'vtv_claude': '**Test labels: ensemble suggestions screened by Claude (not human-checked): accuracy = agreement with them.**\n\n'}.get(TAG, '')
    display(Markdown(note + open(f'reports/transfer_{TAG}_summary.md', encoding='utf-8').read()))
    for m in MODELS:
        run = latest(f'{runs_dir}/{m}_s{SEEDS[0]}_*/test_metrics.json')
        if run and os.path.exists(f'{run}/confusion_matrix.png'):
            display(Image(f'{run}/confusion_matrix.png', width=420))